In [1]:
import os 
import sys 
import torch 
import torch .nn as nn 
import torch .optim as optim 
import numpy as np 
from pathlib import Path 


project_root =Path .cwd ()
while not (project_root /'requirements.txt').exists ():
    if project_root ==project_root .parent :
        raise RuntimeError ("Could not find project root (looked for requirements.txt)")
    project_root =project_root .parent 
sys .path .insert (0 ,str (project_root ))


from src .models .main .classical import ShallowCNN ,ClassicalModel 
from src .models .main .models import QuantumModel ,ExtentedQuantumModel 
from src .models .fusion import OnlineFusionModel 


from src .utils .dataloader import BreastMNISTLoader ,INbreastLoader ,BUSUCLMLoader 


from src .utils .training import train_epoch ,validate_model ,test_model ,training_loop 


device =torch .device ('cuda'if torch .cuda .is_available ()else 'cpu')
print (f"Using device: {device }")
print (f"Project root: {project_root }")

Using device: cuda
Project root: /home/yrsobrinho/HybridFeatureFusionForBreastCancer


In [2]:
DATASET ='BreastMNIST'

CLASSICAL_MODEL_TYPE ='shallow'
QUANTUM_MODEL_TYPE ='quantum'

# CLASSICAL_PRETRAINED_PATH = "/home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/BreastMNIST/classical/shallow/multi-run/best_shallow_no_encoder_run5.pth"
CLASSICAL_PRETRAINED_PATH = "/home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/BreastMNIST/classical/resnet/multi_run/best_resnet_run5.pth"
QUANTUM_PRETRAINED_PATH = "/home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/BreastMNIST/quantum/non-trainable/multi_run/best_quantum_run4.pth"


FREEZE_CLASSICAL = True
FREEZE_QUANTUM = True


BATCH_SIZE =16 
IMAGE_SIZE =28 
NUM_EPOCHS =10 
FEATURE_DIM =128 
NUM_CLASSES =2 

LR_CLASSICAL =1e-4 
LR_QUANTUM =1e-2 
LR_FUSION =1e-3 

N_QUBITS =4 
N_LAYERS =1 
PATCH_SIZE =2 

OUTPUT_DIR =str (project_root /'experiments'/'models'/'fusion'/'online'/DATASET /f'{CLASSICAL_MODEL_TYPE }_{QUANTUM_MODEL_TYPE }')
os .makedirs (OUTPUT_DIR ,exist_ok =True )

print (f"\n{'='*60 }")
print (f"Configuração:")
print (f"  Dataset: {DATASET }")
print (f"  Classical Model: {CLASSICAL_MODEL_TYPE }")
print (f"  Quantum Model: {QUANTUM_MODEL_TYPE }")
print (f"  Feature Dimension: {FEATURE_DIM }")
print (f"  Learning Rates: Classical={LR_CLASSICAL }, Quantum={LR_QUANTUM }, Fusion={LR_FUSION }")
print (f"  Output Directory: {OUTPUT_DIR }")
print (f"{'='*60 }\n")


Configuração:
  Dataset: BreastMNIST
  Classical Model: shallow
  Quantum Model: quantum
  Feature Dimension: 128
  Learning Rates: Classical=0.0001, Quantum=0.01, Fusion=0.001
  Output Directory: /home/yrsobrinho/HybridFeatureFusionForBreastCancer/experiments/models/fusion/online/BreastMNIST/shallow_quantum



In [3]:

if DATASET =='BreastMNIST':
    loader =BreastMNISTLoader (batch_size =BATCH_SIZE ,image_size =IMAGE_SIZE )
elif DATASET =='INbreast':
    loader =INbreastLoader (batch_size =BATCH_SIZE ,image_size =IMAGE_SIZE ,
    root_dir =str (project_root /'src'/'data'/'INbreast'/'png'))
elif DATASET =='BUSUCLM':
    loader =BUSUCLMLoader (batch_size =BATCH_SIZE ,image_size =IMAGE_SIZE ,
    root_dir =str (project_root /'src'/'data'/'BUS-UCLM'/'png'))
else :
    raise ValueError (f"Dataset não suportado: {DATASET }")


train_loader ,test_loader ,val_loader =loader .get_dataloaders ()

print (f"✓ Dataset {DATASET } carregado:")
print (f"  Train batches: {len (train_loader )}")
print (f"  Val batches: {len (val_loader )}")
print (f"  Test batches: {len (test_loader )}")

Using downloaded and verified file: /home/yrsobrinho/.medmnist/breastmnist.npz
Using downloaded and verified file: /home/yrsobrinho/.medmnist/breastmnist.npz
Using downloaded and verified file: /home/yrsobrinho/.medmnist/breastmnist.npz
✓ Dataset BreastMNIST carregado:
  Train batches: 35
  Val batches: 5
  Test batches: 10


In [7]:

if CLASSICAL_MODEL_TYPE =='shallow':
    classical_model =ShallowCNN (num_classes =NUM_CLASSES ,input_channels =1 )
elif CLASSICAL_MODEL_TYPE =='resnet':
    classical_model =ClassicalModel (num_classes =NUM_CLASSES )
else :
    raise ValueError (f"Tipo de modelo clássico não suportado: {CLASSICAL_MODEL_TYPE }")

print (f"✓ Modelo clássico criado: {CLASSICAL_MODEL_TYPE }")


if QUANTUM_MODEL_TYPE =='quantum':

    rand_params =np .random .uniform (high =2 *np .pi ,size =(N_QUBITS ,N_LAYERS ))
    quantum_model =QuantumModel (
    n_qubits =N_QUBITS ,
    n_layers =N_LAYERS ,
    rand_params =rand_params ,
    output_size =(14 ,14 ),
    patch_size =PATCH_SIZE ,
    num_classes =NUM_CLASSES 
    )
elif QUANTUM_MODEL_TYPE =='extended_quantum':

    quantum_model =ExtentedQuantumModel (
    n_qubits =N_QUBITS ,
    n_layers =N_LAYERS ,
    num_classes =NUM_CLASSES ,
    patch_size =PATCH_SIZE ,
    mode ="2d",
    stride =PATCH_SIZE ,
    use_parallel =True 
    )
else :
    raise ValueError (f"Tipo de modelo quântico não suportado: {QUANTUM_MODEL_TYPE }")

print (f"✓ Modelo quântico criado: {QUANTUM_MODEL_TYPE }")


fusion_model =OnlineFusionModel (
classical_model =classical_model ,
quantum_model =quantum_model ,
feature_dim =FEATURE_DIM ,
num_classes =NUM_CLASSES ,
freeze_classical =FREEZE_CLASSICAL ,
freeze_quantum =FREEZE_QUANTUM 
)


if CLASSICAL_PRETRAINED_PATH or QUANTUM_PRETRAINED_PATH :
    fusion_model .load_pretrained_weights (
    classical_path =CLASSICAL_PRETRAINED_PATH ,
    quantum_path =QUANTUM_PRETRAINED_PATH ,
    device =device 
    )

fusion_model =fusion_model .to (device )
print (f"\n✓ Modelo de fusão criado e movido para {device }")


total_params =sum (p .numel ()for p in fusion_model .parameters ())
trainable_params =sum (p .numel ()for p in fusion_model .parameters ()if p .requires_grad )
print (f"  Total de parâmetros: {total_params :,}")
print (f"  Parâmetros treináveis: {trainable_params :,}")

✓ Modelo clássico criado: shallow
✓ Modelo quântico criado: quantum
✓ Loaded classical model weights from /home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/BreastMNIST/classical/resnet/multi_run/best_resnet_run5.pth
✓ Loaded quantum model weights from /home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/BreastMNIST/quantum/non-trainable/multi_run/best_quantum_run4.pth

✓ Modelo de fusão criado e movido para cuda
  Total de parâmetros: 93,892
  Parâmetros treináveis: 514


/home/yrsobrinho/HybridFeatureFusionForBreastCancer/src/models/fusion.py:206: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(classical_path, map_locat

In [8]:

print("Inicializando projeções...")
fusion_model .eval ()
with torch .no_grad ():
    dummy_input =torch .randn (1 ,1 ,IMAGE_SIZE ,IMAGE_SIZE ).to (device )
    _ =fusion_model (dummy_input )
fusion_model .train ()
print ("✓ Projeções inicializadas\n")


param_groups =fusion_model .get_parameter_groups (
lr_classical =LR_CLASSICAL ,
lr_quantum =LR_QUANTUM ,
lr_fusion =LR_FUSION 
)

optimizer =optim .Adam (param_groups )


criterion =nn .CrossEntropyLoss ()

print (f"✓ Otimizador configurado com {len (param_groups )} grupos de parâmetros")
print (f"✓ Loss function: CrossEntropyLoss\n")


group_names =['Clássico','Quântico','Fusão']
for i ,group in enumerate (param_groups ):
    num_params =sum (p .numel ()for p in group ['params'])
    num_tensors =len (group ['params'])
    name =group_names [i ]if i <len (group_names )else f"Grupo {i +1 }"
    print (f"  {name }:")
    print (f"    Learning Rate: {group ['lr']}")
    print (f"    Parâmetros Treináveis: {num_params :,}")
    print (f"    Número de Tensores: {num_tensors }")


print (f"\n{'='*60 }")
print ("DEBUG: Verificação de Parâmetros por Modelo")
print (f"{'='*60 }")

classical_params =sum (p .numel ()for p in fusion_model .classical_model .parameters ()if p .requires_grad )
quantum_params =sum (p .numel ()for p in fusion_model .quantum_model .parameters ()if p .requires_grad )
projection_params =0 
if fusion_model .classical_projection is not None :
    projection_params +=sum (p .numel ()for p in fusion_model .classical_projection .parameters ())
if fusion_model .quantum_projection is not None :
    projection_params +=sum (p .numel ()for p in fusion_model .quantum_projection .parameters ())
fusion_layers_params =sum (p .numel ()for p in fusion_model .fusion_norm .parameters ())
fusion_layers_params +=sum (p .numel ()for p in fusion_model .fusion_fc .parameters ())

print (f"Modelo Clássico: {classical_params :,} parâmetros")
print (f"Modelo Quântico: {quantum_params :,} parâmetros")
print (f"Projeções (classical + quantum): {projection_params :,} parâmetros")
print (f"Camadas de Fusão (norm + fc): {fusion_layers_params :,} parâmetros")
print (f"TOTAL: {classical_params +quantum_params +projection_params +fusion_layers_params :,} parâmetros")
print (f"{'='*60 }\n")

Inicializando projeções...


✓ Projeções inicializadas

✓ Otimizador configurado com 1 grupos de parâmetros
✓ Loss function: CrossEntropyLoss

  Clássico:
    Learning Rate: 0.001
    Parâmetros Treináveis: 117,506
    Número de Tensores: 6

DEBUG: Verificação de Parâmetros por Modelo
Modelo Clássico: 0 parâmetros
Modelo Quântico: 0 parâmetros
Projeções (classical + quantum): 116,992 parâmetros
Camadas de Fusão (norm + fc): 514 parâmetros
TOTAL: 117,506 parâmetros



In [ ]:

history =training_loop (
model =fusion_model ,
train_loader =train_loader ,
val_loader =val_loader ,
criterion =criterion ,
optimizer =optimizer ,
device =device ,
num_epochs =NUM_EPOCHS ,
save_dir =OUTPUT_DIR ,
model_name =f"fusion_{CLASSICAL_MODEL_TYPE }_{QUANTUM_MODEL_TYPE }",
extract_features =False ,
save_best_only =True ,
save_all_epochs =False 
)

print ("\n"+"="*60 )
print ("✓ Treinamento concluído!")
print ("="*60 )

In [ ]:

best_model_path =os .path .join (OUTPUT_DIR ,f"best_fusion_{CLASSICAL_MODEL_TYPE }_{QUANTUM_MODEL_TYPE }.pth")

if os .path .exists (best_model_path ):
    fusion_model .load_state_dict (torch .load (best_model_path ,map_location =device ))
    print (f"✓ Melhor modelo carregado de {best_model_path }")
else :
    print (f"⚠ Arquivo de modelo não encontrado. Usando modelo atual.")


hyperparams ={
'dataset':DATASET ,
'classical_model':CLASSICAL_MODEL_TYPE ,
'quantum_model':QUANTUM_MODEL_TYPE ,
'feature_dim':FEATURE_DIM ,
'batch_size':BATCH_SIZE ,
'lr_classical':LR_CLASSICAL ,
'lr_quantum':LR_QUANTUM ,
'lr_fusion':LR_FUSION ,
'n_qubits':N_QUBITS ,
'n_layers':N_LAYERS ,
'patch_size':PATCH_SIZE 
}

output_file =os .path .join (OUTPUT_DIR ,'test_results.txt')

test_loss ,test_acc ,test_prec ,test_rec ,test_f1 ,test_auc =test_model (
model =fusion_model ,
dataloader =test_loader ,
criterion =criterion ,
device =device ,
hyperparams =hyperparams ,
output_file =output_file ,
extract_features =False ,
compute_detailed_metrics =True 
)

print ("\n"+"="*60 )
print ("RESULTADOS FINAIS NO CONJUNTO DE TESTE")
print ("="*60 )
print (f"Loss: {test_loss :.4f}")
print (f"Accuracy: {test_acc :.4f}")
print (f"Precision: {test_prec :.4f}")
print (f"Recall: {test_rec :.4f}")
print (f"F1 Score: {test_f1 :.4f}")
print (f"AUC: {test_auc :.4f}")
print ("="*60 )
print (f"\nResultados detalhados salvos em: {output_file }")

In [ ]:
import umap 

fusion_model .eval ()
classical_features =[]
quantum_features =[]
fusion_features =[]
sample_labels =[]

with torch .no_grad ():
    for images ,labels in test_loader :
        images =images .to (device )

        classical_output =fusion_model .classical_model .extract_features (images )
        classical_features .append (classical_output .cpu ().numpy ())

        quantum_output =fusion_model .quantum_model .extract_features (images )
        quantum_features .append (quantum_output .cpu ().numpy ())

        fusion_output =fusion_model .extract_features (images )
        fusion_features .append (fusion_output .cpu ().numpy ())

        sample_labels .append (labels .numpy ())

classical_features =np .concatenate (classical_features ,axis =0 )
quantum_features =np .concatenate (quantum_features ,axis =0 )
fusion_features =np .concatenate (fusion_features ,axis =0 )
sample_labels =np .concatenate (sample_labels ,axis =0 ).squeeze ()

print (f"Classical features: {classical_features .shape }")
print (f"Quantum features: {quantum_features .shape }")
print (f"Fusion features: {fusion_features .shape }")
print (f"Labels: {sample_labels .shape }")

reducer =umap .UMAP (n_components =2 ,random_state =42 ,n_neighbors =15 ,min_dist =0.1 )

classical_2d =reducer .fit_transform (classical_features )
plt .figure (figsize =(10 ,8 ))
scatter =plt .scatter (classical_2d [:,0 ],classical_2d [:,1 ],
c =sample_labels ,cmap ='coolwarm',alpha =0.6 ,s =50 )
plt .colorbar (scatter ,label ='Class')
plt .xlabel ('UMAP Component 1')
plt .ylabel ('UMAP Component 2')
plt .title ('UMAP - Classical Features')
plt .grid (True ,alpha =0.3 )
plt .tight_layout ()
plt .savefig (os .path .join (OUTPUT_DIR ,'umap_classical_features.png'),dpi =300 ,bbox_inches ='tight')
plt .show ()
print (f"✓ Classical UMAP saved at {OUTPUT_DIR }/umap_classical_features.png")

quantum_2d =reducer .fit_transform (quantum_features )
plt .figure (figsize =(10 ,8 ))
scatter =plt .scatter (quantum_2d [:,0 ],quantum_2d [:,1 ],
c =sample_labels ,cmap ='coolwarm',alpha =0.6 ,s =50 )
plt .colorbar (scatter ,label ='Class')
plt .xlabel ('UMAP Component 1')
plt .ylabel ('UMAP Component 2')
plt .title ('UMAP - Quantum Features')
plt .grid (True ,alpha =0.3 )
plt .tight_layout ()
plt .savefig (os .path .join (OUTPUT_DIR ,'umap_quantum_features.png'),dpi =300 ,bbox_inches ='tight')
plt .show ()
print (f"✓ Quantum UMAP saved at {OUTPUT_DIR }/umap_quantum_features.png")

fusion_2d =reducer .fit_transform (fusion_features )
plt .figure (figsize =(10 ,8 ))
scatter =plt .scatter (fusion_2d [:,0 ],fusion_2d [:,1 ],
c =sample_labels ,cmap ='coolwarm',alpha =0.6 ,s =50 )
plt .colorbar (scatter ,label ='Class')
plt .xlabel ('UMAP Component 1')
plt .ylabel ('UMAP Component 2')
plt .title ('UMAP - Fused Features')
plt .grid (True ,alpha =0.3 )
plt .tight_layout ()
plt .savefig (os .path .join (OUTPUT_DIR ,'umap_fused_features.png'),dpi =300 ,bbox_inches ='tight')
plt .show ()
print (f"✓ Fusion UMAP saved at {OUTPUT_DIR }/umap_fused_features.png")

In [ ]:
import json 


config ={
'dataset':DATASET ,
'classical_model':CLASSICAL_MODEL_TYPE ,
'quantum_model':QUANTUM_MODEL_TYPE ,
'feature_dim':FEATURE_DIM ,
'num_classes':NUM_CLASSES ,
'batch_size':BATCH_SIZE ,
'image_size':IMAGE_SIZE ,
'num_epochs':NUM_EPOCHS ,
'lr_classical':LR_CLASSICAL ,
'lr_quantum':LR_QUANTUM ,
'lr_fusion':LR_FUSION ,
'n_qubits':N_QUBITS ,
'n_layers':N_LAYERS ,
'patch_size':PATCH_SIZE ,
'freeze_classical':FREEZE_CLASSICAL ,
'freeze_quantum':FREEZE_QUANTUM ,
'total_params':total_params ,
'trainable_params':trainable_params ,
'test_results':{
'loss':float (test_loss ),
'accuracy':float (test_acc ),
'precision':float (test_prec ),
'recall':float (test_rec ),
'f1':float (test_f1 ),
'auc':float (test_auc )
}
}

config_path =os .path .join (OUTPUT_DIR ,'config.json')
with open (config_path ,'w')as f :
    json .dump (config ,f ,indent =4 )

print (f"✓ Configuração salva em {config_path }")
print ("\n"+"="*60 )
print ("Pipeline de Fusão Online Concluída!")
print ("="*60 )

## Multi-run evaluation

Trains the model above **N_RUNS** times from scratch (seeds `0..N_RUNS-1`) and reports **mean ± std** plus a **95% confidence interval** (Student's t) for every test metric, together with per-run and aggregated confusion matrices.

All artefacts are written under `reports/online_classify/` (see `reports/README.md`). Adjust `N_RUNS` below if this is too slow — quantum runs in particular are expensive.

In [9]:
# ==========================================================================
# MULTI-RUN EVALUATION  (N_RUNS independent runs, seeds 0..N_RUNS-1)
# Retrains the model from scratch each run, then reports mean +/- std and a
# 95% CI (Student's t) for every test metric, plus per-run and aggregated
# confusion matrices. Everything is written under `reports/online_classify/`.
# Run the setup cells above first (imports, data loaders, config, model class).
# ==========================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.stats import t as t_dist
from sklearn.metrics import confusion_matrix

N_RUNS = 5
CONFIDENCE_LEVEL = 0.95
NUM_CLASSES = NUM_CLASSES
EPOCHS_PER_RUN = NUM_EPOCHS
RUN_TAG = f'fusion_{CLASSICAL_MODEL_TYPE}_{QUANTUM_MODEL_TYPE}'
CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]

# Output location (rename the subfolder freely; just update the string below).
try:
    _root = str(project_root)
except NameError:
    _root = os.getcwd()
REPORTS_DIR = os.path.join(_root, "reports", "shf")
multi_run_dir = os.path.join(REPORTS_DIR, "resnet+quantum", "multi_run")
os.makedirs(multi_run_dir, exist_ok=True)


def _build_run_model(seed):
    """Return (model, optimizer, model_name) freshly built for this run."""
    if CLASSICAL_MODEL_TYPE == 'shallow':
        cmodel = ShallowCNN(num_classes=NUM_CLASSES, input_channels=1)
    else:
        cmodel = ClassicalModel(num_classes=NUM_CLASSES)
    if QUANTUM_MODEL_TYPE == 'quantum':
        rp = np.random.uniform(high=2 * np.pi, size=(N_QUBITS, N_LAYERS))
        qmodel = QuantumModel(n_qubits=N_QUBITS, n_layers=N_LAYERS, rand_params=rp,
                              output_size=(14, 14), patch_size=PATCH_SIZE, num_classes=NUM_CLASSES)
    else:
        qmodel = ExtentedQuantumModel(n_qubits=N_QUBITS, n_layers=N_LAYERS, num_classes=NUM_CLASSES,
                                      patch_size=PATCH_SIZE, mode="2d", stride=PATCH_SIZE, use_parallel=True)
    fmodel = OnlineFusionModel(classical_model=cmodel, quantum_model=qmodel,
                               feature_dim=FEATURE_DIM, num_classes=NUM_CLASSES,
                               freeze_classical=FREEZE_CLASSICAL, freeze_quantum=FREEZE_QUANTUM)
    if CLASSICAL_PRETRAINED_PATH or QUANTUM_PRETRAINED_PATH:
        fmodel.load_pretrained_weights(classical_path=CLASSICAL_PRETRAINED_PATH,
                                       quantum_path=QUANTUM_PRETRAINED_PATH, device=device)
    fmodel = fmodel.to(device)
    fmodel.eval()
    with torch.no_grad():
        _ = fmodel(torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE).to(device))  # init lazy projections
    fmodel.train()
    pg = fmodel.get_parameter_groups(lr_classical=LR_CLASSICAL, lr_quantum=LR_QUANTUM, lr_fusion=LR_FUSION)
    opt = optim.Adam(pg)
    name = f"fusion_{CLASSICAL_MODEL_TYPE}_{QUANTUM_MODEL_TYPE}_run{seed + 1}"
    return fmodel, opt, name

def _get_predictions(model, dataloader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            y_pred.extend(outputs.argmax(dim=1).cpu().numpy().tolist())
            y_true.extend(np.asarray(labels).squeeze().reshape(-1).tolist())
    return np.array(y_true), np.array(y_pred)


def _plot_confusion_matrix(cm, save_path, title, class_names=None, normalize=False):
    cm = np.asarray(cm, dtype=float)
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        disp = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)
        cell_fmt = lambda v: f"{v:.1%}"
    else:
        disp = cm
        cell_fmt = lambda v: f"{int(round(v))}"
    n = cm.shape[0]
    names = class_names if class_names is not None else [str(i) for i in range(n)]
    fig, ax = plt.subplots(figsize=(1.4 * n + 2.5, 1.4 * n + 2.5))
    im = ax.imshow(disp, cmap="Blues", vmin=0)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(names, rotation=45, ha="right"); ax.set_yticklabels(names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    thresh = disp.max() / 2 if disp.max() > 0 else 0.5
    for i in range(n):
        for j in range(n):
            ax.text(j, i, cell_fmt(disp[i, j]), ha="center", va="center",
                    color="white" if disp[i, j] > thresh else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


run_results = []
run_confusion_matrices = []

for run_idx in range(N_RUNS):
    seed = run_idx
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    print(f"\n{'=' * 80}")
    print(f"RUN {run_idx + 1}/{N_RUNS} (seed={seed}) - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'=' * 80}\n")

    run_model, run_optimizer, run_model_name = _build_run_model(seed)
    run_criterion = nn.CrossEntropyLoss()

    best_path = os.path.join(multi_run_dir, f"best_{run_model_name}.pth")
    if os.path.isfile(best_path):
        os.remove(best_path)

    training_loop(
        model=run_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=run_criterion,
        optimizer=run_optimizer,
        device=device,
        num_epochs=EPOCHS_PER_RUN,
        save_dir=multi_run_dir,
        model_name=run_model_name,
        extract_features=False,
        save_best_only=True,
        save_all_epochs=False,
    )

    if os.path.isfile(best_path):
        run_model.load_state_dict(torch.load(best_path, map_location=device))
    else:
        print(f"⚠ Warning: run {run_idx + 1} never beat the initial val_acc baseline — "
              f"testing with final-epoch weights instead of a best checkpoint.")
    run_model.eval()

    test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = test_model(
        model=run_model,
        dataloader=test_loader,
        criterion=run_criterion,
        device=device,
        hyperparams={"dataset": DATASET, "classical_model": CLASSICAL_MODEL_TYPE,
                    "quantum_model": QUANTUM_MODEL_TYPE, "feature_dim": FEATURE_DIM,
                    "batch_size": BATCH_SIZE, "lr_classical": LR_CLASSICAL,
                    "lr_quantum": LR_QUANTUM, "lr_fusion": LR_FUSION,
                    "n_qubits": N_QUBITS, "n_layers": N_LAYERS, "patch_size": PATCH_SIZE},
        output_file=os.path.join(multi_run_dir, f"test_log_run{run_idx + 1}.txt"),
        extract_features=False,
        compute_detailed_metrics=True,
    )

    run_labels, run_preds = _get_predictions(run_model, test_loader, device)
    run_cm = confusion_matrix(run_labels, run_preds, labels=list(range(NUM_CLASSES)))
    run_confusion_matrices.append(run_cm)
    _plot_confusion_matrix(
        run_cm,
        save_path=os.path.join(multi_run_dir, f"confusion_matrix_run{run_idx + 1}.png"),
        title=f"Confusion Matrix - Run {run_idx + 1}",
        class_names=CLASS_NAMES,
    )

    run_record = {
        "run": run_idx + 1,
        "loss": test_loss,
        "accuracy": test_acc,
        "precision": test_prec,
        "recall": test_rec,
        "f1": test_f1,
        "auc": test_auc,
    }
    _gamma = getattr(run_model, "gamma", None)
    if _gamma is not None:
        run_record["gamma"] = float(_gamma.item()) if hasattr(_gamma, "item") else float(_gamma)
    run_results.append(run_record)

    print(f"\n✓ Run {run_idx + 1} test accuracy: {test_acc:.4f}")

print(f"\n{'=' * 80}")
print(f"All {N_RUNS} runs complete")
print(f"{'=' * 80}")

metrics_keys = ["loss", "accuracy", "precision", "recall", "f1", "auc"]
if all("gamma" in r for r in run_results):
    metrics_keys.append("gamma")

stats = {}
for key in metrics_keys:
    values = np.array([r[key] for r in run_results], dtype=float)
    n = len(values)
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if n > 1 else 0.0
    if n > 1:
        sem = std / np.sqrt(n)
        t_crit = float(t_dist.ppf((1 + CONFIDENCE_LEVEL) / 2, df=n - 1))
        margin = t_crit * sem
    else:
        margin = 0.0
    stats[key] = {
        "mean": mean, "std": std,
        "min": float(np.min(values)), "max": float(np.max(values)),
        "ci_lower": mean - margin, "ci_upper": mean + margin,
    }

cm_sum = np.stack(run_confusion_matrices).sum(axis=0)
cm_sum_path = os.path.join(multi_run_dir, "confusion_matrix_sum.png")
_plot_confusion_matrix(cm_sum.astype(int), cm_sum_path,
                       f"Confusion Matrix - Sum over {N_RUNS} runs", CLASS_NAMES)
cm_mean_path = os.path.join(multi_run_dir, "confusion_matrix_mean_normalized.png")
_plot_confusion_matrix(cm_sum, cm_mean_path,
                       f"Confusion Matrix - Row-Normalized (mean over {N_RUNS} runs)",
                       CLASS_NAMES, normalize=True)

stats_path = os.path.join(REPORTS_DIR, f"multi_run_statistics_{RUN_TAG}.txt")
report_meta = {
    "Dataset": DATASET,
    "Classical Model": CLASSICAL_MODEL_TYPE,
    "Quantum Model": QUANTUM_MODEL_TYPE,
    "Feature Dim": FEATURE_DIM,
    "Epochs per run": NUM_EPOCHS,
    "Batch Size": BATCH_SIZE,
    "LR (classical/quantum/fusion)": f"{LR_CLASSICAL}/{LR_QUANTUM}/{LR_FUSION}",
}
with open(stats_path, "w", encoding="utf-8") as f:
    f.write("=" * 80 + "\n")
    f.write(f"MULTI-RUN STATISTICS ({N_RUNS} runs)\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    for _k, _v in report_meta.items():
        f.write(f"{_k}: {_v}\n")
    f.write("\n")

    f.write("PER-RUN TEST RESULTS\n")
    f.write("-" * 80 + "\n")
    header = f"{'Run':<5}{'Loss':<10}{'Accuracy':<10}{'Precision':<12}{'Recall':<10}{'F1':<10}{'AUC':<10}"
    if "gamma" in metrics_keys:
        header += f"{'Gamma':<10}"
    f.write(header + "\n")
    for r in run_results:
        line = (f"{r['run']:<5}{r['loss']:<10.4f}{r['accuracy']:<10.4f}"
                f"{r['precision']:<12.4f}{r['recall']:<10.4f}{r['f1']:<10.4f}{r['auc']:<10.4f}")
        if "gamma" in metrics_keys:
            line += f"{r['gamma']:<10.4f}"
        f.write(line + "\n")
    f.write("\n")

    f.write(f"SUMMARY STATISTICS (mean +/- std, min, max, {int(CONFIDENCE_LEVEL * 100)}% CI [t-distribution])\n")
    f.write("-" * 80 + "\n")
    for key in metrics_keys:
        s = stats[key]
        f.write(f"{key.capitalize():<12}: {s['mean']:.4f} +/- {s['std']:.4f}"
                f"  (min: {s['min']:.4f}, max: {s['max']:.4f})"
                f"  {int(CONFIDENCE_LEVEL * 100)}% CI: [{s['ci_lower']:.4f}, {s['ci_upper']:.4f}]\n")
    f.write("\n")

    labels_for_header = CLASS_NAMES if CLASS_NAMES is not None else [str(i) for i in range(cm_sum.shape[0])]
    f.write(f"CONFUSION MATRIX - SUMMED OVER {N_RUNS} RUNS (rows = true, cols = predicted)\n")
    f.write("-" * 80 + "\n")
    f.write("            " + "".join(f"{lbl:>12}" for lbl in labels_for_header) + "\n")
    for lbl, row in zip(labels_for_header, cm_sum.astype(int)):
        f.write(f"{lbl:>12}" + "".join(f"{v:>12d}" for v in row) + "\n")
    f.write(f"\nSaved plot: {cm_sum_path}\n\n")

    f.write(f"CONFUSION MATRIX - ROW-NORMALIZED MEAN OVER {N_RUNS} RUNS\n")
    f.write("-" * 80 + "\n")
    cm_norm = cm_sum.astype(float) / cm_sum.sum(axis=1, keepdims=True)
    f.write("            " + "".join(f"{lbl:>12}" for lbl in labels_for_header) + "\n")
    for lbl, row in zip(labels_for_header, cm_norm):
        f.write(f"{lbl:>12}" + "".join(f"{v:>12.2%}" for v in row) + "\n")
    f.write(f"\nSaved plot: {cm_mean_path}\n")

    f.write("\n" + "=" * 80 + "\n")
    f.write("END OF REPORT\n")
    f.write("=" * 80 + "\n")

print(f"✓ Multi-run statistics saved to: {stats_path}")
print()
for key in metrics_keys:
    s = stats[key]
    print(f"  {key.capitalize():<12}: {s['mean']:.4f} +/- {s['std']:.4f}"
          f"  (min: {s['min']:.4f}, max: {s['max']:.4f})"
          f"  {int(CONFIDENCE_LEVEL * 100)}% CI: [{s['ci_lower']:.4f}, {s['ci_upper']:.4f}]")
print("\nSummed confusion matrix:")
print(cm_sum.astype(int))



RUN 1/5 (seed=0) - 2026-07-27 17:20:29

✓ Loaded classical model weights from /home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/BreastMNIST/classical/resnet/multi_run/best_resnet_run5.pth
✓ Loaded quantum model weights from /home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/BreastMNIST/quantum/non-trainable/multi_run/best_quantum_run4.pth


Epoch 1/10



Epoch 1 Summary:
  Train - Loss: 0.6991, Acc: 0.5751
  Val   - Loss: 0.8327, Acc: 0.3462, Precision: 0.8000, Recall: 0.1404, F1: 0.2388, AUC: 0.7176
  ✓ New best model saved (val_acc: 0.3462) at /home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/shf/resnet+quantum/multi_run/best_fusion_shallow_quantum_run1.pth
  ✓ Quantum rand_params sidecar saved at /home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/shf/resnet+quantum/multi_run/best_fusion_shallow_quantum_run1_rand_params.npy
Epoch 2/10



Epoch 2 Summary:
  Train - Loss: 0.6342, Acc: 0.6337
  Val   - Loss: 0.5743, Acc: 0.7308, Precision: 0.8750, Recall: 0.7368, F1: 0.8000, AUC: 0.7828
  ✓ New best model saved (val_acc: 0.7308) at /home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/shf/resnet+quantum/multi_run/best_fusion_shallow_quantum_run1.pth
  ✓ Quantum rand_params sidecar saved at /home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/shf/resnet+quantum/multi_run/best_fusion_shallow_quantum_run1_rand_params.npy
Epoch 3/10



Epoch 3 Summary:
  Train - Loss: 0.5798, Acc: 0.7106
  Val   - Loss: 0.5646, Acc: 0.7308, Precision: 0.9286, Recall: 0.6842, F1: 0.7879, AUC: 0.8321
Epoch 4/10


...


Epoch 9 Summary:
  Train - Loss: 0.5521, Acc: 0.7179
  Val   - Loss: 0.5095, Acc: 0.7821, Precision: 0.8448, Recall: 0.8596, F1: 0.8522, AUC: 0.7836
Epoch 10/10


/tmp/ipykernel_2966709/2344023186.py:137: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  run_model.load_state_dict(torch.load(best_path, map_location=device))



Epoch 10 Summary:
  Train - Loss: 0.5002, Acc: 0.7601
  Val   - Loss: 0.5710, Acc: 0.6923, Precision: 0.8511, Recall: 0.7018, F1: 0.7692, AUC: 0.7744


Testing: 100%|██████████| 10/10 [00:57<00:00,  5.77s/it]


Test Loss: 0.4730
Accuracy: 0.7692, Precision: 0.7708, Recall: 0.9737, F1: 0.8605, AUC: 0.7928
Macro - Precision: 0.7604, Recall: 0.5940, F1: 0.5969
Micro - Precision: 0.7692, Recall: 0.7692, F1: 0.7692

✓ Run 5 test accuracy: 0.7692

All 5 runs complete
✓ Multi-run statistics saved to: /home/yrsobrinho/HybridFeatureFusionForBreastCancer/reports/shf/multi_run_statistics_fusion_shallow_quantum.txt

  Loss        : 0.4766 +/- 0.0238  (min: 0.4425, max: 0.5092)  95% CI: [0.4471, 0.5062]
  Accuracy    : 0.7846 +/- 0.0289  (min: 0.7500, max: 0.8269)  95% CI: [0.7488, 0.8205]
  Precision   : 0.8355 +/- 0.0432  (min: 0.7708, max: 0.8866)  95% CI: [0.7819, 0.8891]
  Recall      : 0.8860 +/- 0.0895  (min: 0.7544, max: 0.9737)  95% CI: [0.7749, 0.9970]
  F1          : 0.8561 +/- 0.0270  (min: 0.8152, max: 0.8907)  95% CI: [0.8227, 0.8896]
  Auc         : 0.8114 +/- 0.0124  (min: 0.7928, max: 0.8231)  95% CI: [0.7960, 0.8269]

Summed confusion matrix:
[[107 103]
 [ 65 505]]
